<a href="https://colab.research.google.com/github/ainankhan-pixel/231A052_NLP_Lab/blob/main/nlpPROJECTexp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Ainan Khan 231A052
!pip install -q streamlit scikit-learn networkx

In [3]:
%%writefile app.py
 #Ainan Khan 231A052
import streamlit as st
import re
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def split_sentences(text):
    """Split text into sentences without requiring NLTK."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def summarize_text(text, num_sentences):

    sentences = split_sentences(text)

    if len(sentences) <= num_sentences:
        return text

    # Convert sentences into TF-IDF vectors
    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Calculate similarity between sentences
    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Create graph
    graph = nx.from_numpy_array(similarity_matrix)

    # PageRank
    scores = nx.pagerank(graph)

    # Rank sentences
    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Select important sentences
    selected_indices = [
        index for index, score in ranked[:num_sentences]
    ]

    # Keep original article order
    selected_indices.sort()

    summary = " ".join(
        sentences[index]
        for index in selected_indices
    )

    return summary


# -----------------------------
# STREAMLIT APP
# -----------------------------

st.set_page_config(
    page_title="InShortAI",
    page_icon="📰",
    layout="wide"
)

st.title("📰 InShortAI")

st.subheader("NLP-Based Text Summarization")

st.write(
    "Enter a long article and generate a short summary "
    "using the TextRank algorithm."
)

text = st.text_area(
    "Enter your article:",
    height=300,
    placeholder="Paste your article here..."
)

num_sentences = st.slider(
    "Number of summary sentences:",
    1,
    10,
    3
)

if st.button("✨ Summarize"):

    if not text.strip():

        st.warning("Please enter some text.")

    else:

        sentences = split_sentences(text)

        if len(sentences) < 2:

            st.warning(
                "Please enter at least 2 sentences."
            )

        else:

            summary = summarize_text(
                text,
                num_sentences
            )

            st.subheader("📌 Summary")

            st.write(summary)

            # Word counts
            original_words = len(
                text.split()
            )

            summary_words = len(
                summary.split()
            )

            compression = (
                1 - summary_words / original_words
            ) * 100

            col1, col2, col3 = st.columns(3)

            col1.metric(
                "Original Words",
                original_words
            )

            col2.metric(
                "Summary Words",
                summary_words
            )

            col3.metric(
                "Compression",
                f"{compression:.1f}%"
            )

Writing app.py


In [8]:
#Ainan Khan 231A052
!ls -l app.py

In [5]:
#Ainan Khan 231A052
!streamlit run app.py --server.address=0.0.0.0 --server.port=8501 > /content/streamlit.log 2>&1 &

In [6]:
#Ainan Khan 231A052
!pip install -q pyngrok
from pyngrok import ngrok

try:
    ngrok.set_auth_token("3GxahrNjZhfmwnjgKcAeqML49TS_4G28ijstCJCBXY6U8JFXr")
except Exception as e:
    if "already set" in str(e):
        print("ngrok auth token is already configured!")
    else:
        raise e

In [7]:
#Ainan Khan 231A052
public_url = ngrok.connect(8501)
print(public_url)


NgrokTunnel: "https://refocus-pacifier-numbly.ngrok-free.dev" -> "http://localhost:8501"
